# 03 · LoRA from scratch

Write LoRA yourself in ~20 lines of PyTorch: the math `W′ = W + (α/r)·B·A`, why `B` starts at zero, how few
parameters it trains, how to merge it back for free inference, and how it compares with full fine-tuning on a real
(small) adaptation task.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min, no downloads · **Needs:**
[PyTorch essentials](../04_numpy_pandas_pytorch/03_pytorch_essentials.ipynb),
[01 · When to fine-tune](01_when_to_fine_tune_and_data_prep.ipynb) (the fine-tuning menu)

## Why this matters in interviews

- "What is LoRA and why does it work?" is the single most-asked fine-tuning question. The strong answer has the
  formula, the zero-init of `B`, the parameter arithmetic and the merge — all four are in this notebook, as code.
- Follow-ups are numeric: "how many parameters at r = 8 on a 4096×4096 layer?", "which modules do you target?",
  "does it slow inference?". You will compute each one.
- "Implement a LoRA layer" is a realistic live-coding drill.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tn04` | What is LoRA and why does it work? |
| `tn14` | What hyperparameters matter for LoRA, and how do you set them? |
| `tn06` | When would you choose full fine-tuning over LoRA? |
| `tn26` | What is the difference between prompt tuning, prefix tuning and LoRA? |
| `tn34` | How do you serve many fine-tuned models without paying for many GPUs? |
| `tr45` | You are fine-tuning this model with LoRA. Which modules go in target_modules, and how many weights will you train? |
| `tn12` | What is catastrophic forgetting and how do you avoid it? |
| `fo33` | What is transfer learning, and how does it relate to fine-tuning? |

In [ ]:
import copy, math, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.set_num_threads(1)
torch.manual_seed(0); np.random.seed(0)

## 1. The idea and the math

**In plain English:** instead of editing a huge weight matrix, you freeze it and learn a small *correction* next to
it. The correction is forced to be "thin" — the product of a tall matrix and a wide one — so it has very few numbers.
The bet: the change needed to adapt a pretrained model to a narrow task is simple (low-rank), even though the matrix
itself is not.

For a frozen weight `W` (shape `d_out × d_in`), LoRA learns `B` (`d_out × r`) and `A` (`r × d_in`) with `r ≪ d`:

```
h = W x + (α / r) · B (A x)        equivalently   W′ = W + (α / r) · B A
```

- `A` starts small and random, **`B` starts at zero**, so `B A = 0`: training starts *exactly* at the base model.
- Trainable parameters: `r · (d_in + d_out)` instead of `d_in · d_out`.
- `α / r` is a fixed scale; `α` (`lora_alpha`) sets how strongly the update is applied.
- After training, add `(α/r)·B A` into `W` once, and the adapter disappears — no extra inference cost.

```
          x ───────────────► W (frozen, d_out × d_in) ────────────┐
          │                                                       (+) ──► h
          └──► A (r × d_in) ──► B (d_out × r) ──► × α/r ──────────┘
                trainable          trainable, starts at 0
```

## 2. A `LoRALinear` module

In [ ]:
class LoRALinear(nn.Module):
    """Wraps a frozen nn.Linear and adds a trainable low-rank update (alpha/r) * B @ A."""

    def __init__(self, base: nn.Linear, r: int = 8, alpha: float = 16, dropout: float = 0.0):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        for p in self.base.parameters():
            p.requires_grad_(False)                                   # the pretrained weight never changes
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))              # same init PEFT uses for A
        self.B = nn.Parameter(torch.zeros(base.out_features, r))      # zero => B @ A = 0 at step 0
        self.dropout = nn.Dropout(dropout)
        self.enabled = True

    def forward(self, x):
        out = self.base(x)
        if self.enabled:
            out = out + self.scale * (self.dropout(x) @ self.A.T @ self.B.T)
        return out

    def merged(self) -> nn.Linear:
        """A plain nn.Linear with W' = W + scale * B @ A folded in."""
        lin = nn.Linear(self.base.in_features, self.base.out_features, bias=self.base.bias is not None)
        with torch.no_grad():
            lin.weight.copy_(self.base.weight + self.scale * self.B @ self.A)
            if self.base.bias is not None:
                lin.bias.copy_(self.base.bias)
        return lin

layer = nn.Linear(512, 512)
lora = LoRALinear(copy.deepcopy(layer), r=8, alpha=16)
x = torch.randn(4, 512)
print("max |difference| at step 0:", (lora(x) - layer(x)).abs().max().item())
assert torch.equal(lora(x), layer(x))                    # B = 0: identical output before any training

n_train = sum(p.numel() for p in lora.parameters() if p.requires_grad)
n_frozen = sum(p.numel() for p in lora.parameters() if not p.requires_grad)
print(f"trainable: {n_train:,} = r*(d_in+d_out) = 8*(512+512)  |  frozen: {n_frozen:,}  ({n_train / n_frozen:.1%})")
assert n_train == 8 * (512 + 512)

Why `B = 0` and not `A = 0`? If both were zero, the gradient of each would be zero too (each gradient contains the
other matrix) and nothing would ever train. With `A` random and `B` zero, `B` gets a non-zero gradient on the first
step, `A` starts moving one step later, and the model still begins exactly at the pretrained function. Check the
first gradients:

In [ ]:
lora.B.grad = lora.A.grad = None
loss = lora(x).pow(2).mean()
loss.backward()
print(f"step-0 gradient norm  B: {lora.B.grad.norm():.4f}   A: {lora.A.grad.norm():.4f}")
assert lora.B.grad.norm() > 0 and lora.A.grad.norm() == 0

## 3. How many parameters? (the arithmetic interviewers ask for)

For one 4096×4096 layer (a 7B model's attention projection), full fine-tuning trains 16.8M numbers; LoRA trains
`r · 8192`.

In [ ]:
d = 4096
ranks = [1, 2, 4, 8, 16, 32, 64]
full_layer = d * d
for r in ranks:
    print(f"r={r:>2}: {r * (d + d):>9,} trainable  = {r * (d + d) / full_layer:6.2%} of {full_layer:,}")

# a Llama-2-7B-shaped model: 32 layers, hidden 4096, MLP 11008
L, d_ff, full_7b = 32, 11008, 6_738_415_616
shapes = {"q_proj": (d, d), "k_proj": (d, d), "v_proj": (d, d), "o_proj": (d, d),
          "gate_proj": (d, d_ff), "up_proj": (d, d_ff), "down_proj": (d_ff, d)}
count = lambda r, mods: L * sum(r * (shapes[m][0] + shapes[m][1]) for m in mods)
qv, all7 = ["q_proj", "v_proj"], list(shapes)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
a1.semilogy(ranks, [full_layer] * len(ranks), "k--", label="full fine-tuning (16.8M)")
a1.semilogy(ranks, [r * 2 * d for r in ranks], "o-", label="LoRA r·(d_in + d_out)")
a1.set_xlabel("rank r"); a1.set_ylabel("trainable parameters (log)"); a1.set_title("One 4096×4096 layer"); a1.legend()
a2.semilogy(ranks, [full_7b] * len(ranks), "k--", label="full fine-tuning (6.7B)")
a2.semilogy(ranks, [count(r, all7) for r in ranks], "o-", label="LoRA on all 7 Linear / layer")
a2.semilogy(ranks, [count(r, qv) for r in ranks], "s-", label="LoRA on q_proj + v_proj")
a2.set_xlabel("rank r"); a2.set_title("A 7B model (32 layers)"); a2.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"7B, r=16, all 7 Linear: {count(16, all7):,} ({count(16, all7) / full_7b:.2%});  r=8, q+v: {count(8, qv):,} ({count(8, qv) / full_7b:.3%})")
assert count(64, all7) / full_7b < 0.03

Even at r = 64 on every Linear layer, LoRA trains under 3% of a 7B model. That changes the memory bill completely:
no gradients or Adam states for the frozen 6.7B (see the memory arithmetic in
[05 · QLoRA](05_qlora_with_bitsandbytes_and_peft.ipynb)), and the saved adapter is tens of MB instead of 13 GB — small
enough to keep one per customer and hot-swap them on one base model (`tn34`).

## 4. A real adaptation: task A → task B

**Setup.** "Pretrain" a small MLP on **task A**: classify 8×8 handwritten digits (scikit-learn's bundled `digits`).
**Task B** is the same digits **rotated 90°** — same labels, shifted input distribution, a stand-in for "same
skill, new domain". Then adapt the task-A model to task B two ways:

- **full fine-tuning**: every weight trains;
- **LoRA**: the MLP is frozen, a `LoRALinear` wraps each of its three Linear layers (r = 4, α = 8).

We measure accuracy on B (did it learn?), accuracy on A (did it forget?) and trainable parameters.

In [ ]:
X, y = load_digits(return_X_y=True)
X = X / 16.0
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
rotate = lambda X_: np.ascontiguousarray(np.rot90(X_.reshape(-1, 8, 8), 1, axes=(1, 2))).reshape(-1, 64)
XtrB, XteB = rotate(Xtr), rotate(Xte)
T = lambda a: torch.tensor(a, dtype=torch.float32)

fig, axes = plt.subplots(2, 6, figsize=(9, 3.2))
for k in range(6):
    axes[0, k].imshow(Xte[k].reshape(8, 8), cmap="gray_r"); axes[0, k].set_title(f"A: {yte[k]}", fontsize=9)
    axes[1, k].imshow(XteB[k].reshape(8, 8), cmap="gray_r"); axes[1, k].set_title(f"B: {yte[k]}", fontsize=9)
for ax in axes.ravel(): ax.axis("off")
fig.suptitle("Task A: digits   ·   Task B: the same digits rotated 90°"); plt.tight_layout(); plt.show()

In [ ]:
def make_mlp():
    return nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 10))

def fit(model, X_, y_, epochs, lr, batch_size=64, seed=0):
    g = torch.Generator().manual_seed(seed)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    X_, y_ = T(X_), torch.tensor(y_)
    for _ in range(epochs):
        perm = torch.randperm(len(X_), generator=g)
        for i in range(0, len(X_), batch_size):
            idx = perm[i:i + batch_size]
            loss = F.cross_entropy(model(X_[idx]), y_[idx])
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def accuracy(model, X_, y_):
    with torch.no_grad():
        return (model(T(X_)).argmax(1).numpy() == y_).mean()

n_trainable = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)

torch.manual_seed(0)
t0 = time.time()
pretrained = fit(make_mlp(), Xtr, ytr, epochs=20, lr=2e-3)
print(f"'pretrained' on task A in {time.time() - t0:.1f}s")
print(f"task A accuracy: {accuracy(pretrained, Xte, yte):.3f}   task B accuracy (never seen): {accuracy(pretrained, XteB, yte):.3f}")

In [ ]:
def add_lora(model, r, alpha):
    m = copy.deepcopy(model)
    for p in m.parameters():
        p.requires_grad_(False)
    for i, mod in enumerate(m):
        if isinstance(mod, nn.Linear):
            m[i] = LoRALinear(mod, r=r, alpha=alpha)
    return m

torch.manual_seed(1)
full_ft = fit(copy.deepcopy(pretrained), XtrB, ytr, epochs=20, lr=1e-3)
torch.manual_seed(1)
lora_ft = fit(add_lora(pretrained, r=4, alpha=8), XtrB, ytr, epochs=20, lr=3e-3)     # LoRA likes a higher LR

def set_adapters(model, on):
    for mod in model:
        if isinstance(mod, LoRALinear):
            mod.enabled = on

results = {"task-A model (no tuning)": (pretrained, 0),
           "full fine-tuning": (full_ft, n_trainable(full_ft)),
           "LoRA r=4": (lora_ft, n_trainable(lora_ft))}
rows = {}
for name, (m, n) in results.items():
    rows[name] = (accuracy(m, XteB, yte), accuracy(m, Xte, yte), n)
set_adapters(lora_ft, False)
rows["LoRA r=4, adapter switched off"] = (accuracy(lora_ft, XteB, yte), accuracy(lora_ft, Xte, yte), 0)
set_adapters(lora_ft, True)
print(f"{'':32s} {'acc B':>6} {'acc A':>6} {'trainable':>10}")
for name, (b, a, n) in rows.items():
    print(f"{name:32s} {b:6.3f} {a:6.3f} {n:>10,}")
assert rows["LoRA r=4, adapter switched off"][1] == rows["task-A model (no tuning)"][1]   # base untouched

In [ ]:
names = list(rows)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [2, 1]})
xs = np.arange(len(names))
a1.bar(xs - 0.2, [rows[n][0] for n in names], 0.4, label="task B (new)", color="#3b5bdb")
a1.bar(xs + 0.2, [rows[n][1] for n in names], 0.4, label="task A (old)", color="#adb5bd")
a1.set_xticks(xs); a1.set_xticklabels([n.replace(", ", ",\n") for n in names], fontsize=8)
a1.set_ylabel("test accuracy"); a1.set_title("Learning B vs remembering A"); a1.legend()
a2.bar(["full FT", "LoRA r=4"], [rows["full fine-tuning"][2], rows["LoRA r=4"][2]], color=["#e8590c", "#2f9e44"])
a2.set_yscale("log"); a2.set_ylabel("trainable parameters (log)"); a2.set_title("What each one trains")
plt.tight_layout(); plt.show()

**Read the result.**

- Full fine-tuning learns B best — it can move all ~26k weights — and in doing so damages A (catastrophic
  forgetting, notebook [07](07_distillation_and_catastrophic_forgetting.ipynb)).
- LoRA with about 11× fewer trainable parameters gets most of the way on B. With the adapter **on**, A also degrades —
  the adapter changes the function for every input — but in this run less than with full fine-tuning.
- With the adapter **off**, task A is back to *exactly* the original accuracy. That is LoRA's real guarantee: the base
  weights are never modified, so the original model is always one switch away, and many adapters can share one base.

## 5. Merge: zero extra inference cost

Fold `(α/r)·B A` into each `W` and replace the `LoRALinear` wrappers with plain `nn.Linear` layers. The outputs must be
identical (up to float rounding).

In [ ]:
merged = nn.Sequential(*[mod.merged() if isinstance(mod, LoRALinear) else mod for mod in lora_ft])
xb = T(XteB)
with torch.no_grad():
    diff = (merged(xb) - lora_ft(xb)).abs().max().item()
print(f"max |merged - adapter| over {len(xb)} test inputs: {diff:.2e}")
print(f"merged model: {sum(isinstance(m, LoRALinear) for m in merged)} LoRA layers left, "
      f"accuracy B {accuracy(merged, XteB, yte):.3f} (same as before)")
assert diff < 1e-5 and accuracy(merged, XteB, yte) == rows["LoRA r=4"][0]

After merging, the model is an ordinary network of the original size: no extra matmuls, no latency. Keep adapters
**unmerged** only when you want to hot-swap them per request (multi-LoRA serving: one base, many tenants).

## 6. Rank sweep: how much capacity does B need?

In [ ]:
sweep_ranks = [1, 2, 4, 8, 16]
t0 = time.time()
sweep = {}
for r in sweep_ranks:
    torch.manual_seed(1)
    m = fit(add_lora(pretrained, r=r, alpha=2 * r), XtrB, ytr, epochs=20, lr=3e-3)
    sweep[r] = (accuracy(m, XteB, yte), accuracy(m, Xte, yte), n_trainable(m))
print(f"{len(sweep_ranks)} LoRA runs in {time.time() - t0:.1f}s")
for r, (b, a, n) in sweep.items():
    print(f"r={r:>2}: acc B {b:.3f}  acc A (adapter on) {a:.3f}  trainable {n:>6,}")

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.semilogx(sweep_ranks, [sweep[r][0] for r in sweep_ranks], "o-", base=2, label="LoRA: task B accuracy")
ax.axhline(rows["full fine-tuning"][0], color="#e8590c", ls="--", label="full fine-tuning: task B")
ax.semilogx(sweep_ranks, [sweep[r][1] for r in sweep_ranks], "s:", base=2, color="#868e96", label="LoRA: task A (adapter on)")
ax.set_xlabel("rank r (α = 2r)"); ax.set_ylabel("test accuracy"); ax.set_title("Rank sweep: diminishing returns on the new task")
ax.legend(fontsize=8); plt.show()
assert sweep[16][0] > sweep[1][0]

Accuracy on B climbs quickly with rank and then flattens toward full fine-tuning: the extra capacity stops paying.
The grey line is the price: higher rank moves the model further, so task A (adapter on) degrades steadily — and at
the larger ranks in this run LoRA forgets *more* than full fine-tuning did (with its 3× higher learning rate).
"LoRA learns less and forgets less" is a statement about *small* ranks and gentle settings, not a guarantee; the only
guarantee is the one from section 4 — switch the adapter off and the base is untouched. In LLM practice r = 8–16 is typical for style/format, 32–64 for
bigger behaviour changes, and the choice of **target modules** matters more than the rank (`tn14`).

## 7. The α/r scaling, explained

`scale = α / r`. Two reasons it exists:

1. **Tuning convenience.** With `α` fixed relative to `r` (the common convention `α = 2r`, so `scale = 2`), you can
   change the rank without re-tuning the learning rate much.
2. **Learning speed across ranks.** `A` is random, so the adapter's path `x → A x` carries more signal as `r`
   grows. How much a single step *improves the loss* grows roughly like `scale² · r`: with the standard `α/r`
   and a fixed α that is `α²/r`, so higher ranks learn *slower*; with `α/√r` it is `α²`, independent of the rank.
   The measurement below uses one plain SGD step on a random regression target (Adam normalises gradients, which
   changes the constants but not the idea).

In [ ]:
torch.manual_seed(0)
x_ = torch.randn(256, 512)
target = torch.randn(256, 512)
alpha = 16

def first_step_gain(r, scale, lr=0.01):
    torch.manual_seed(r)
    lay = LoRALinear(nn.Linear(512, 512), r=r, alpha=alpha); lay.scale = scale
    before = F.mse_loss(lay(x_), target)
    before.backward()
    with torch.no_grad():
        lay.B -= lr * lay.B.grad                              # one SGD step on B (A's gradient is 0 at step 0)
        after = F.mse_loss(lay(x_), target)
    return (before - after).item()                            # how much that one step reduced the loss

rs = [1, 4, 16, 64, 256]
std = [first_step_gain(r, alpha / r) for r in rs]
rs_lora = [first_step_gain(r, alpha / math.sqrt(r)) for r in rs]
for r, a, b in zip(rs, std, rs_lora):
    print(f"r={r:>3}: loss drop after one step   α/r: {a:.2e}   α/√r (rsLoRA): {b:.2e}")

plt.figure(figsize=(7, 3.4))
plt.loglog(rs, std, "o-", base=2, label="scale = α/r (standard LoRA)")
plt.loglog(rs, rs_lora, "s-", base=2, label="scale = α/√r (rsLoRA)")
plt.xlabel("rank r"); plt.ylabel("loss reduction from one SGD step"); plt.legend()
plt.title("Fixed α: standard scaling learns slower at high rank; α/√r does not"); plt.show()
assert std[-1] < std[0] / 20 and max(rs_lora) / min(rs_lora) < 3

With a *fixed* α, the standard `α/r` scale makes high-rank adapters learn slower, so they under-train unless you
raise α or the learning rate — one reason people reported "no gain above r = 64". That is also why the convention
`α = 2r` exists: it keeps `α/r` constant as you change `r`. **rsLoRA** (rank-stabilised LoRA) uses `α/√r`, which
keeps the learning speed roughly rank-independent (PEFT: `LoraConfig(use_rslora=True)`).

## 8. Where LoRA goes in a transformer

Every decoder layer has seven Linear layers. For a 7B model (hidden 4096, MLP 11008), per layer at rank r:

| module | shape (out × in) | LoRA params at r = 16 | notes |
|---|---|---|---|
| `q_proj`, `k_proj`, `v_proj`, `o_proj` | 4096 × 4096 | 131,072 each | attention; q + v was the original LoRA paper's choice |
| `gate_proj`, `up_proj` | 11008 × 4096 | 241,664 each | MLP: ~2/3 of each layer's weights live here |
| `down_proj` | 4096 × 11008 | 241,664 | |
| norms, rotary embeddings, activations | – | – | nothing (or nothing useful) to adapt |
| `embed_tokens`, `lm_head` | vocab × hidden | large | only if you add tokens; beware tied weights |

Rule of thumb: all attention projections beat q + v, and attention + MLP (the QLoRA paper's recommendation) usually
beats attention-only, because most of the network's capacity is in the MLP. In PEFT this is just
`target_modules=[...]` or `"all-linear"` — see [04 · LoRA with PEFT](04_lora_with_peft.ipynb) for a real model's
module names.

## 9. LoRA variants (brief, hedged)

| Variant | Idea in one line | When people reach for it |
|---|---|---|
| **DoRA** | split W into magnitude × direction; LoRA updates the direction, magnitude learned separately | often a small quality gain over LoRA at the same rank; a bit slower (`use_dora=True` in PEFT) |
| **LoRA+** | give B a larger learning rate than A (ratio ~16 in the paper) | faster / better convergence with the same parameters |
| **rsLoRA** | scale by α/√r instead of α/r | high ranks (64+) that otherwise under-train (`use_rslora=True`) |
| **VeRA** | freeze shared random A and B across layers; train only small scaling vectors | extreme parameter savings (10×+ fewer than LoRA), some quality cost |
| **PiSSA / LoftQ** (init tricks) | initialise A, B from the top singular vectors of W (PiSSA) or to offset quantisation error (LoftQ) | faster convergence; LoftQ pairs with quantised bases |
| **AdaLoRA** | adapt the rank per layer during training (prune unimportant directions) | a fixed parameter budget spread where it matters |

Gains reported for these are usually modest and task-dependent — run plain LoRA first as the baseline.

## Try it yourself

**1.** Make LoRA wrap only the **last** Linear layer (the classifier head). How does task-B accuracy compare with LoRA
on all three layers? What does that say about where the needed change lives?

In [ ]:
# Solution — try it yourself first, then run this
head_only = copy.deepcopy(pretrained)
for p in head_only.parameters():
    p.requires_grad_(False)
head_only[4] = LoRALinear(head_only[4], r=4, alpha=8)
torch.manual_seed(1)
fit(head_only, XtrB, ytr, epochs=20, lr=3e-3)
print(f"LoRA on the head only : acc B {accuracy(head_only, XteB, yte):.3f}, trainable {n_trainable(head_only):,}")
print(f"LoRA on all 3 layers  : acc B {rows['LoRA r=4'][0]:.3f}, trainable {rows['LoRA r=4'][2]:,}")
print("Rotation changes which pixels matter, so the first layer's features must change: adapting the head is not enough.")

**2.** Verify the parameter formula for a rectangular layer: wrap `nn.Linear(4096, 11008)` (an MLP `up_proj`) at
r = 16 and count trainable parameters. Compare with `r * (d_in + d_out)`.

In [ ]:
# Solution — try it yourself first, then run this
up = LoRALinear(nn.Linear(4096, 11008, bias=False), r=16, alpha=32)
print(f"counted: {n_trainable(up):,}   formula: {16 * (4096 + 11008):,}   full layer: {4096 * 11008:,}")
assert n_trainable(up) == 16 * (4096 + 11008)

**3.** Two adapters, one base: train a second LoRA on task A-with-**horizontal-flip** and serve both tasks from the
same frozen MLP by swapping only `A` and `B`. Check that the base weights are shared (same tensor).

In [ ]:
# Solution — try it yourself first, then run this
flip = lambda X_: np.ascontiguousarray(X_.reshape(-1, 8, 8)[:, :, ::-1]).reshape(-1, 64)
torch.manual_seed(2)
lora_flip = fit(add_lora(pretrained, r=4, alpha=8), flip(Xtr), ytr, epochs=20, lr=3e-3)
adapters = {name: [(m.A.detach().clone(), m.B.detach().clone()) for m in model if isinstance(m, LoRALinear)]
            for name, model in [("rotate", lora_ft), ("flip", lora_flip)]}
server = add_lora(pretrained, r=4, alpha=8)                       # one copy of the base

def load_adapter(name):
    for mod, (A_, B_) in zip([m for m in server if isinstance(m, LoRALinear)], adapters[name]):
        mod.A.data.copy_(A_); mod.B.data.copy_(B_)

for name, Xtest in [("rotate", XteB), ("flip", flip(Xte))]:
    load_adapter(name)
    print(f"adapter {name:6s}: accuracy {accuracy(server, Xtest, yte):.3f}")
kb = sum(a.numel() + b.numel() for a, b in adapters['flip']) * 4 / 1024
print(f"each adapter is {kb:.0f} KB; the base ({sum(p.numel() for p in pretrained.parameters()) * 4 / 1024:.0f} KB) is stored once")

## Say it in an interview

- **30 seconds:** "LoRA freezes the pretrained weight W and learns a low-rank update B·A beside it, scaled by α/r.
  B starts at zero, so training starts exactly at the base model. It trains r·(d_in + d_out) parameters per layer —
  65k instead of 16.8M for a 4096² layer at r = 8, well under 1% of a 7B model — so there are no gradients or
  optimizer states for the base. After training you merge B·A into W and inference costs nothing extra; or keep
  adapters separate and hot-swap them per tenant on one base."
- **Why it works:** the update needed to adapt a pretrained model to a narrow task has low intrinsic rank — you steer
  existing capability rather than build new capability.
- **Hyperparameters:** r 8–16 (style/format) to 32–64 (bigger changes); α = 2r convention (or rsLoRA's α/√r at high
  rank); LR ~1e-4–3e-4, about 10× a full fine-tune; target modules matter most — attention + MLP beats q + v.
- **Traps:** "does LoRA slow inference?" (no, if merged); "does LoRA prevent forgetting?" (the base is untouched and
  recoverable exactly; with the adapter on, forgetting is smaller at low rank but not zero); zero-initialising both
  matrices (nothing trains); a full-fine-tune learning rate on LoRA (nothing moves).
- **When full fine-tuning instead (`tn06`):** 50k+ clean examples, a target far from pretraining (new language or
  modality), multi-GPU budget — and still run a LoRA baseline first.

## Next

- [04 · LoRA with PEFT](04_lora_with_peft.ipynb) — the same thing on a real 135M chat model, with the library
- [05 · QLoRA with bitsandbytes and PEFT](05_qlora_with_bitsandbytes_and_peft.ipynb) — LoRA on a 4-bit base
- [07 · Distillation and catastrophic forgetting](07_distillation_and_catastrophic_forgetting.ipynb) — forgetting and its mitigations
- [Neuron and backprop from scratch](../06_deep_learning/01_neuron_and_backprop_from_scratch.ipynb) · [Implement-from-scratch drills](../17_interview_coding_rounds/07_implement_from_scratch_drills.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)